In [19]:
import pandas as pd

from utils.base.pipelines import PredictionPipeline
from utils.experiment.core import (
    compute_bucket_statistics,
    get_config_hash,
    load_config,
)
from utils.experiment.naming import (
    build_output_folder_name,
    get_dataset_name,
    get_model_name,
    get_timestamp,
)
from utils.factories import (
    create_bucketer,
    create_dataset,
    create_model,
    create_transformer,
)


In [20]:
config_path = 'C:/Users/Pavel/Desktop/PROJECTS/master-thesis/conf/experiments/offline/outcome/logistic_regression/BPIC_12_O.yaml'

config = load_config(config_path)

In [21]:
dataset_name = get_dataset_name(config)
model_name = get_model_name(config)
config_hash = get_config_hash(config)
timestamp = get_timestamp()
run_id = build_output_folder_name(
    dataset_name=dataset_name,
    model_name=model_name,
    config_hash=config_hash,
    fallback_name='run',
    timestamp=timestamp,
)

print('=' * 60)
print(f'Run ID: {run_id}')
print('=' * 60)

print(f'Config: {config_path}')
print(f'Dataset: {dataset_name}')
print(f'Transformer: {config["transformer"]["type"]}')
print(f'Model: {model_name}')

# Create dataset and load data
dataset = create_dataset(config['dataset'])

train_df, test_df = dataset.train_test_split()

train_prefixes, y_train = dataset.get_prefixes_and_labels(train_df)
test_prefixes, y_test = dataset.get_prefixes_and_labels(test_df)

num_classes = pd.concat([y_train, y_test]).nunique()

# Create pipeline components
bucketer = create_bucketer(config['bucketer'])
transformer = create_transformer(config['transformer'])
model = create_model(config['model'], device='cpu', num_classes=num_classes)

# Train pipeline
pipeline = PredictionPipeline(bucketer, transformer, model)

pipeline.fit(train_prefixes, y_train)

# Generate predictions
y_train_pred = pipeline.predict(train_prefixes)
y_test_pred = pipeline.predict(test_prefixes)

# Compute statistics
train_bucket_stats_df = compute_bucket_statistics(
    train_prefixes, y_train, y_train_pred, bucketer
)
test_bucket_stats_df = compute_bucket_statistics(
    test_prefixes, y_test, y_test_pred, bucketer
)

Run ID: bpic_12_o_logistic_regression_0fb922a5_2026-06-10_12-23-23
Config: C:/Users/Pavel/Desktop/PROJECTS/master-thesis/conf/experiments/offline/outcome/logistic_regression/BPIC_12_O.yaml
Dataset: BPIC_12_O
Transformer: aggregate
Model: logistic_regression


c:\Users\Pavel\Desktop\PROJECTS\master-thesis\.venv\lib\site-packages\pm4py\utils.py:1000: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn(
c:\Users\Pavel\Desktop\PROJECTS\master-thesis\.venv\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
c:\Users\Pavel\Desktop\PROJECTS\master-thesis\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 5015/5015 [00:00<00:00, 5971.26it/s]


Training bucket: 1...
Training bucket: 2...
Training bucket: 3...
Training bucket: 4...
Training bucket: 5...
Training bucket: 6...
Training bucket: 7...
Training bucket: 8...
Training bucket: 9...
Training bucket: 10...
Training bucket: 11...
Training bucket: 12...
Training bucket: 13...
Training bucket: 14...
Training bucket: 15...
Training bucket: 16...
Training bucket: 17...
Training bucket: 18...
Training bucket: 19...
Training bucket: 20...
Training bucket: 21...
Training bucket: 22...
Training bucket: 23...
Training bucket: 24...
Training bucket: 25...
Training bucket: 26...
Training bucket: 27...
Training bucket: 28...
Training bucket: 29...
Training bucket: 30...


In [23]:
test_bucket_stats_df

,bucket,prefixes,events,accuracy,precision,recall,f1_macro
0,1,1690,1690,0.527811,0.278584,0.527811,0.172734
1,2,1690,4268,0.626054,0.391944,0.626054,0.192507
2,3,1690,6013,0.601863,0.362239,0.601863,0.187863
3,4,1539,7099,0.840541,0.873542,0.840541,0.443587
4,5,1178,6833,1.000000,1.000000,1.000000,1.000000
5,6,559,4297,1.000000,1.000000,1.000000,1.000000
6,7,559,4856,1.000000,1.000000,1.000000,1.000000
7,8,495,4903,1.000000,1.000000,1.000000,1.000000
8,9,380,4363,1.000000,1.000000,1.000000,1.000000
9,10,208,416,1.000000,1.000000,1.000000,1.000000


## Synthetic example

In [1]:
import uuid

import numpy as np
import pandas as pd


TS_COL = 'time:timestamp'
CASE_COL = 'case:concept:name'
ACTIVITY_COL = 'concept:name'


def traces_to_log_df(trace_fn, *args, start_timestamp='2020-01-01', **kwargs):
    traces = trace_fn(*args, **kwargs)

    records = []
    timestamp = pd.Timestamp(start_timestamp)

    for _, case_activities in enumerate(traces):
        case_id = str(uuid.uuid4())[-8:]
        for activity in case_activities:
            timestamp += pd.Timedelta(hours=1)
            records.append(
                {
                    CASE_COL: case_id,
                    ACTIVITY_COL: activity,
                    TS_COL: timestamp,
                }
            )

    return pd.DataFrame(records)


def get_traces(num_cases):
    traces = []
    for _ in range(num_cases):
        if np.random.rand() > 0.5:
            traces.append(['A', 'B', 'C', 'D', 'E'])
        else:
            traces.append(['A', 'B', 'X', 'Y', 'Z'])
    return traces


def get_labels(log_df):
    records = []
    for case_id, row in log_df.groupby(CASE_COL, sort=False):
        last_activity = row[ACTIVITY_COL].iloc[-1]
        if last_activity == 'E':
            outcome = 1
        elif last_activity == 'Z':
            outcome = 0
        else:
            continue

        records.append({CASE_COL: case_id, 'outcome': outcome})

    return pd.DataFrame(records)


In [2]:
log_df = traces_to_log_df(get_traces, num_cases=10)
labels_df = get_labels(log_df)

In [3]:
log_df.head()

,case:concept:name,concept:name,time:timestamp
0,9a87586b,A,2020-01-01 01:00:00
1,9a87586b,B,2020-01-01 02:00:00
2,9a87586b,C,2020-01-01 03:00:00
3,9a87586b,D,2020-01-01 04:00:00
4,9a87586b,E,2020-01-01 05:00:00


In [4]:
from torch import optim
from sklearn.ensemble import RandomForestClassifier

from utils.base.pipelines import PredictionPipeline
from utils.base.bucketers import PrefixLengthBucketer, NoBucketer
from utils.base.transformers import IndexBasedTransformer
from utils.base.datasets import NextActivityDataset, OutcomeDataset
from utils.base.feature_extractors import (
    CalendarFeatureExtractor,
    ProcessContextFeatureExtractor,
    TemporalFeatureExtractor,
)
from utils.constants import CASE_PREFIX_COL
from utils.experiment.core import compute_bucket_statistics
from models.wrapper import TorchModelWrapper, SklearnModelWrapper
from models.lstm import LSTM

In [5]:
dataset = NextActivityDataset(
    raw_df=log_df,
    train_ratio=0.8,
    feature_extractors=[
        # TemporalFeatureExtractor()
    ]
)

In [8]:
dataset = OutcomeDataset(
    raw_df=log_df,
    labels_df=labels_df,
    train_ratio=0.8,
    feature_extractors=[
        # TemporalFeatureExtractor(),
        # CalendarFeatureExtractor(),
        # ProcessContextFeatureExtractor(),
    ]
)

In [9]:
train_df, test_df = dataset.train_test_split()

train_prefixes, y_train = dataset.get_prefixes_and_labels(train_df)
test_prefixes, y_test = dataset.get_prefixes_and_labels(test_df)

num_classes = pd.concat([y_train, y_test]).nunique()

In [10]:
bucketer = PrefixLengthBucketer()
transformer = IndexBasedTransformer(
    cat_cols=[
        'concept:name'
    ]
)

model = SklearnModelWrapper(
    model=RandomForestClassifier(n_estimators=100, random_state=42)
)

pipeline = PredictionPipeline(bucketer, transformer, model=model)

pipeline.fit(train_prefixes, y_train)

Training bucket: 1...
Training bucket: 2...
Training bucket: 3...
Training bucket: 4...
Training bucket: 5...


In [11]:
y_train_pred = pipeline.predict(train_prefixes)
y_test_pred = pipeline.predict(test_prefixes)

In [12]:
train_bucket_stats_df = compute_bucket_statistics(
    train_prefixes, y_train, y_train_pred, bucketer
)
test_bucket_stats_df = compute_bucket_statistics(
    test_prefixes, y_test, y_test_pred, bucketer
)

In [13]:
train_bucket_stats_df

,bucket,prefixes,events,accuracy,precision,recall,f1_macro
0,1,8,8,0.75,0.5625,0.75,0.428571
1,2,8,16,0.75,0.5625,0.75,0.428571
2,3,8,24,1.00,1.0000,1.00,1.000000
3,4,8,32,1.00,1.0000,1.00,1.000000
4,5,8,40,1.00,1.0000,1.00,1.000000


In [14]:
test_bucket_stats_df

,bucket,prefixes,events,accuracy,precision,recall,f1_macro
0,1,2,2,0.5,0.25,0.5,0.333333
1,2,2,4,0.5,0.25,0.5,0.333333
2,3,2,6,1.0,1.00,1.0,1.000000
3,4,2,8,1.0,1.00,1.0,1.000000
4,5,2,10,1.0,1.00,1.0,1.000000


In [ ]:
bucketer = PrefixLengthBucketer()
transformer = IndexBasedTransformer()

model = SklearnModelWrapper(
    model=RandomForestClassifier(n_estimators=100, random_state=42)
)

pipeline = PredictionPipeline(bucketer, transformer, model=model)

pipeline.fit(train_prefixes, y_train)

Training bucket: 1...


IndexError: Boolean index has wrong length: 120 instead of 112

In [ ]:
num_classes = len(dataset.label_encoder.classes_)

bucketer = NoBucketer()
transformer = IndexBasedTransformer()

model = TorchModelWrapper(
    model=LSTM(
        input_dim=,
        num_classes=num_classes,
        hidden_dim=64,
        dropout=0,
        num_layers=1,
    ),
    optimizer_cls=lambda p: optim.Adam(p, {'lr': 0.001}),
    loss_fn=
    epochs=,
    batch_size=,
    device=
)

pipeline = PredictionPipeline(bucketer, transformer, model=None)

pipeline.fit(train_prefixes, y_train)